# LAB 03 — Delta Lake Fundamentals: DML, Time Travel & RESTORE

**Databricks Fundamentals | ~60 min | Difficulty: ★★★★☆**

---

## Scenario
You manage an orders Delta table in a retail data platform.
You need to implement full lifecycle management: upserts, corrections,
historical auditing and disaster recovery.

## Task overview
| # | Topic | Fill-in? |
|---|-------|----------|
| 01 | Setup & create demo table | given |
| 02 | INSERT batch of rows | **TODO** |
| 03 | UPDATE with condition | **TODO** |
| 04 | DELETE with condition | **TODO** |
| 05 | MERGE INTO — upsert pattern | **TODO** |
| 06 | DESCRIBE HISTORY — audit trail | given |
| 07 | Time Travel — VERSION AS OF | **TODO** |
| 08 | Time Travel — TIMESTAMP AS OF | **TODO** |
| 09 | RESTORE TABLE | **TODO** |
| 10 | Schema evolution — add column with mergeSchema | **TODO** |
| 11 | VACUUM — storage cleanup | **TODO** |
| 12 | Advanced MERGE — full SCD1 with DELETE clause | **TODO** |


In [ ]:
%run ../setup/00_setup

In [ ]:
# Task 01 — Create demo table (GIVEN)

TABLE = f"{CATALOG}.{SILVER_SCHEMA}.lab_orders_delta"

spark.sql(f"DROP TABLE IF EXISTS {TABLE}")

spark.sql(f"""
    CREATE TABLE {TABLE} (
        order_id     STRING  NOT NULL,
        customer_id  STRING,
        status       STRING,
        total_amount DOUBLE,
        order_date   DATE
    )
    USING DELTA
""")

# Seed data
spark.sql(f"""
    INSERT INTO {TABLE} VALUES
        ('O001', 'C001', 'pending',   120.00, '2025-01-10'),
        ('O002', 'C002', 'shipped',   340.50, '2025-01-11'),
        ('O003', 'C001', 'pending',    85.00, '2025-01-12'),
        ('O004', 'C003', 'delivered', 210.00, '2025-01-13'),
        ('O005', 'C002', 'pending',   500.00, '2025-01-14')
""")

spark.table(TABLE).show()
print(f"Table created: {TABLE}")


In [ ]:
# Task 02 — INSERT a new batch of orders
# TODO: insert the following rows into the table using a single INSERT INTO ... VALUES statement
#   ('O006', 'C004', 'pending',   99.00, '2025-01-15')
#   ('O007', 'C005', 'shipped',  440.00, '2025-01-15')
#   ('O008', 'C001', 'pending',  175.00, '2025-01-16')

spark.sql(f"""
    -- TODO: write INSERT INTO {TABLE} VALUES ( ... )
""")

print(f"Row count after insert: ", spark.table(TABLE).count())


In [ ]:
# Task 03 — UPDATE with condition
# Business rule: orders with status='pending' older than 2025-01-13 are now 'cancelled'
# TODO: write the UPDATE statement

spark.sql(f"""
    -- TODO: UPDATE {TABLE}
    -- SET    status = 'cancelled'
    -- WHERE  status = 'pending'
    -- AND    order_date < '2025-01-13'
""")

print("After UPDATE:")
spark.table(TABLE).orderBy("order_id").show()


In [ ]:
# Task 04 — DELETE with condition
# Remove all orders for customer C005 (test account — data loaded in error)
# TODO: write the DELETE statement

spark.sql(f"""
    -- TODO: DELETE FROM {TABLE} WHERE customer_id = 'C005'
""")

print(f"Row count after DELETE: ", spark.table(TABLE).count())
spark.table(TABLE).orderBy("order_id").show()


In [ ]:
# Task 05 — MERGE INTO (upsert)
# A corrections file arrives with updated totals and a new order.
# Rules:
#   WHEN MATCHED (by order_id) → update total_amount and status from source
#   WHEN NOT MATCHED           → insert the new row

from datetime import date
from pyspark.sql import Row

corrections = [
    Row(order_id='O002', customer_id='C002', status='returned',  total_amount=0.0,    order_date=date(2025,1,11)),
    Row(order_id='O004', customer_id='C003', status='delivered', total_amount=220.00, order_date=date(2025,1,13)),
    Row(order_id='O009', customer_id='C006', status='pending',   total_amount=310.00, order_date=date(2025,1,17)),
]
df_corrections = spark.createDataFrame(corrections)
df_corrections.createOrReplaceTempView("corrections")

spark.sql(f"""
    -- TODO: write MERGE INTO {TABLE} AS tgt
    --       USING corrections AS src
    --       ON    tgt.order_id = src.order_id
    --       WHEN MATCHED     THEN UPDATE SET ...
    --       WHEN NOT MATCHED THEN INSERT  VALUES (...)
""")

print("After MERGE:")
spark.table(TABLE).orderBy("order_id").show()


In [ ]:
# Task 06 — DESCRIBE HISTORY (GIVEN)

spark.sql(f"DESCRIBE HISTORY {TABLE}").select(
    "version", "timestamp", "operation", "operationParameters", "operationMetrics"
).show(truncate=False)

# ❓ How many versions exist? What operation created each version?
# ❓ Which version performed the MERGE? How many rows were inserted / updated?


In [ ]:
# Task 07 — Time Travel: VERSION AS OF
# TODO: read the table at version 0 (the initial seed state) and show all rows
# Hint: spark.read.format("delta").option("versionAsOf", N).table(...)
#   OR:  spark.sql("SELECT * FROM table VERSION AS OF N")

df_v0 = spark.sql(f"""
    -- TODO: SELECT * FROM {TABLE} VERSION AS OF 0
""")

print("State at VERSION 0 (seed data):")
df_v0.orderBy("order_id").show()

# ❓ How many rows? Which statuses appear?


In [ ]:
# Task 08 — Time Travel: TIMESTAMP AS OF
# TODO:
#   1. Read DESCRIBE HISTORY to get the timestamp of version 1 (after first INSERT batch)
#   2. Re-read the table at that timestamp using .option("timestampAsOf", ...)

# Step 1 — get version 1 timestamp (given helper)
history = spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version", "timestamp")
v1_ts = history.filter("version = 1").collect()[0]["timestamp"]
print(f"Version 1 timestamp: {v1_ts}")

# Step 2 — TODO: read the table at v1_ts
df_v1 = (
    spark.read
         .format("delta")
         # TODO: .option("timestampAsOf", str(v1_ts))
         # TODO: .table(TABLE)
)

print("State at version 1 timestamp:")
df_v1.orderBy("order_id").show()


In [ ]:
# Task 09 — RESTORE TABLE
# Scenario: A rogue UPDATE corrupted the data. You need to roll back to version 1.
# TODO: use RESTORE TABLE ... TO VERSION AS OF 1

spark.sql(f"""
    -- TODO: RESTORE TABLE {TABLE} TO VERSION AS OF 1
""")

print("After RESTORE to version 1:")
spark.table(TABLE).orderBy("order_id").show()

# Verify with DESCRIBE HISTORY — a new version is created for the RESTORE
spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version","operation","operationParameters").show(truncate=False)


In [ ]:
# Task 10 — Schema evolution: add a new column
# Business requirement: add a 'region' column (String) to existing records.
# Existing rows should have region = 'EMEA' (default).
# New rows coming in will carry the region value.

# 10a — create a new batch DataFrame with the extra column
from pyspark.sql import functions as F

new_batch = spark.createDataFrame([
    ("O010", "C007", "pending", 130.0, "2025-01-18", "APAC"),
    ("O011", "C008", "shipped", 290.0, "2025-01-18", "AMER"),
], ["order_id", "customer_id", "status", "total_amount", "order_date", "region"])

new_batch = new_batch.withColumn("order_date", F.to_date("order_date"))

# 10b — TODO: write new_batch to the table using mode="append" and mergeSchema=True
# Without mergeSchema=True this will fail — try it first, then add the option.

(
    new_batch.write
    .format("delta")
    .mode("append")
    # TODO: .option("mergeSchema", "true")
    .saveAsTable(TABLE)
)

print("Schema after evolution:")
spark.table(TABLE).printSchema()
spark.table(TABLE).orderBy("order_id").show(truncate=False)


In [ ]:
# Task 11 — VACUUM
# VACUUM removes old data files that are no longer referenced by any Delta version.
# ⚠ After VACUUM you can NO LONGER time-travel to vacuumed versions.

# 11a — check current number of Parquet files (given)
import subprocess
detail = spark.sql(f"DESCRIBE DETAIL {TABLE}").collect()[0]
print(f"Location: {detail['location']}")
print(f"numFiles: {detail['numFiles']}")
print(f"sizeInBytes: {detail['sizeInBytes']}")

# 11b — TODO: run VACUUM with 0-hour retention (for demo only — NEVER in production!)
# First you must disable the safety check:
spark.conf.set("spark.databricks.delta.retentionDurationCheck.enabled", "false")

spark.sql(f"""
    -- TODO: VACUUM {TABLE} RETAIN 0 HOURS
""")

# 11c — re-check
print("After VACUUM:")
spark.sql(f"DESCRIBE DETAIL {TABLE}").select("numFiles","sizeInBytes").show()

# ❓ How many files remain?
# ❓ What happens if you try to time-travel to version 0 now?


In [ ]:
# Task 12 — Advanced MERGE: full SCD1 with DELETE clause
# Scenario: a full snapshot arrives each night.
# Rules:
#   WHEN MATCHED AND status changes  → UPDATE
#   WHEN NOT MATCHED IN source       AND target.order_date < '2025-01-15' → DELETE (purge old)
#   WHEN NOT MATCHED IN target       → INSERT

# This uses the MERGE ... WHEN NOT MATCHED BY SOURCE syntax (Databricks Delta extension)

tonight_snapshot = spark.createDataFrame([
    ("O006", "C004", "delivered", 99.00,  "2025-01-15"),
    ("O009", "C006", "shipped",   310.00, "2025-01-17"),
    ("O010", "C007", "delivered", 130.00, "2025-01-18"),
    ("O011", "C008", "delivered", 290.00, "2025-01-18"),
    ("O012", "C009", "pending",   200.00, "2025-01-19"),  # new row
], ["order_id", "customer_id", "status", "total_amount", "order_date"])

tonight_snapshot = tonight_snapshot.withColumn(
    "order_date", F.to_date("order_date")
)
tonight_snapshot.createOrReplaceTempView("tonight_snapshot")

spark.sql(f"""
    -- TODO: MERGE INTO {TABLE} AS tgt
    --       USING tonight_snapshot AS src
    --       ON    tgt.order_id = src.order_id
    --
    -- WHEN MATCHED AND tgt.status <> src.status
    --   THEN UPDATE SET tgt.status       = src.status,
    --                   tgt.total_amount = src.total_amount
    --
    -- WHEN NOT MATCHED
    --   THEN INSERT (order_id, customer_id, status, total_amount, order_date)
    --        VALUES (src.order_id, src.customer_id, src.status, src.total_amount, src.order_date)
    --
    -- WHEN NOT MATCHED BY SOURCE AND tgt.order_date < '2025-01-15'
    --   THEN DELETE
""")

print("Final table state:")
spark.table(TABLE).orderBy("order_id").show(truncate=False)

spark.sql(f"DESCRIBE HISTORY {TABLE}").select("version","operation","operationMetrics").show(1, truncate=False)
